# 12｜CNN 与 TinyViT 公平对照实验准备

本实验只改变网络结构，其余训练条件保持一致。目标不是证明某个架构永远更好，而是在当前 CIFAR-10 任务和同一训练方案下，观察 CNN 与 TinyViT 的差异。

## 1. 控制变量

| 项目 | TinyViT | CNN Baseline |
|---|---:|---:|
| 数据集 | CIFAR-10 | 相同 |
| 训练/验证划分 | 45000 / 5000，seed=42 | 相同函数 |
| 测试集 | 官方 10000 张 | 相同 |
| 数据增强 | Crop、Flip、RandAugment、Erasing | 相同 |
| batch size | 128 | 128 |
| epoch 上限 | 300 | 300 |
| 损失函数 | CrossEntropy + smoothing=0.05 | 相同函数 |
| 优化器 | AdamW | 相同函数 |
| 初始学习率 | 4e-4 | 4e-4 |
| weight decay | 0.05 | 0.05 |
| Mixup alpha | 0.1 | 0.1 |
| Warmup | 10 epochs | 10 epochs |
| 学习率策略 | 余弦退火到 1e-6 | 相同函数 |
| Early Stopping | patience=50，min_delta=1e-5 | 相同函数 |
| 梯度裁剪 | max norm=1.0 | 相同函数 |
| 模型选择 | 验证准确率优先，loss 平局判断 | 相同 `fit()` |

checkpoint 与结果目录必须不同，否则 CNN 会覆盖 TinyViT 的训练结果。

In [1]:
import torch

from cnn_baseline import CNNBaseline, CNN_CHECKPOINT_PATH, CNN_HISTORY_PATH
from vit import (
    BATCH_SIZE, EARLY_STOPPING_MIN_DELTA, EARLY_STOPPING_PATIENCE,
    LABEL_SMOOTHING, LEARNING_RATE, MAX_GRAD_NORM, MIN_LEARNING_RATE,
    MIXUP_ALPHA, NUM_EPOCHS, RANDOM_SEED, WARMUP_EPOCHS, WEIGHT_DECAY,
    TinyViT, create_cifar10_dataloaders, create_training_components,
)

device = torch.device("cpu")
print("本课验证设备：", device)

本课验证设备： cpu


## 2. 唯一主动改变的变量：网络结构

CNN 使用四个阶段，每个阶段包含：

```text
Conv 3x3 → BatchNorm → GELU → Conv 3x3 → BatchNorm → GELU → MaxPool
```

通道逐步增加为 `64 → 128 → 256 → 512`，高宽逐步降低为 `32 → 16 → 8 → 4 → 2`，最后经过全局平均池化和 10 类输出层。

GELU 和 Dropout=0.1 与 TinyViT 保持一致；BatchNorm、卷积、池化属于 CNN 本身的结构差异。

In [2]:
cnn = CNNBaseline().to(device).eval()
images = torch.randn(2, 3, 32, 32, device=device)
stage_shapes = []
hooks = [
    stage.register_forward_hook(
        lambda module, inputs, output: stage_shapes.append(tuple(output.shape))
    )
    for stage in cnn.stages
]
with torch.inference_mode():
    logits = cnn(images)
for hook in hooks:
    hook.remove()

print("输入：", tuple(images.shape))
for index, shape in enumerate(stage_shapes, start=1):
    print(f"Stage {index}：{shape}")
print("输出：", tuple(logits.shape))

assert stage_shapes == [
    (2, 64, 16, 16), (2, 128, 8, 8),
    (2, 256, 4, 4), (2, 512, 2, 2),
]
assert logits.shape == (2, 10)

输入： (2, 3, 32, 32)
Stage 1：(2, 64, 16, 16)
Stage 2：(2, 128, 8, 8)
Stage 3：(2, 256, 4, 4)
Stage 4：(2, 512, 2, 2)
输出： (2, 10)


## 3. 控制模型容量

模型参数量不能完全代表容量，但如果差距几十倍，结构对比会混入明显的规模差异。因此 CNN 的通道数经过设计，使其参数量与当前 TinyViT 相差不超过 2%。

In [3]:
vit = TinyViT()
cnn_parameters = sum(p.numel() for p in cnn.parameters() if p.requires_grad)
vit_parameters = sum(p.numel() for p in vit.parameters() if p.requires_grad)
difference_ratio = abs(cnn_parameters - vit_parameters) / vit_parameters

print(f"CNN 参数量：     {cnn_parameters:,}")
print(f"TinyViT 参数量： {vit_parameters:,}")
print(f"相对差距：       {difference_ratio * 100:.3f}%")
assert difference_ratio < 0.02

CNN 参数量：     4,692,426
TinyViT 参数量： 4,771,082
相对差距：       1.649%


## 4. 数据与训练组件确实来自同一入口

CNN 训练脚本没有重新实现 Dataset、loss、optimizer 或 scheduler，而是直接导入 TinyViT 已使用的函数。下面实际创建一次，检查数据数量和主要参数。

In [4]:
train_loader, validation_loader, test_loader, class_names = (
    create_cifar10_dataloaders(batch_size=BATCH_SIZE, num_workers=0)
)
criterion, optimizer, scheduler = create_training_components(cnn)

print("训练/验证/测试：", len(train_loader.dataset), len(validation_loader.dataset), len(test_loader.dataset))
print("类别：", class_names)
print("损失函数：", criterion)
print("优化器：", optimizer.__class__.__name__)
print("基础学习率：", LEARNING_RATE)
print("Warmup 第 1 轮实际学习率：", optimizer.param_groups[0]["lr"])
print("调度器：", scheduler.__class__.__name__)

assert (len(train_loader.dataset), len(validation_loader.dataset), len(test_loader.dataset)) == (45000, 5000, 10000)
assert criterion.label_smoothing == LABEL_SMOOTHING
assert optimizer.__class__.__name__ == "AdamW"
assert optimizer.param_groups[0]["weight_decay"] == WEIGHT_DECAY

训练/验证/测试： 45000 5000 10000
类别： ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
损失函数： CrossEntropyLoss()
优化器： AdamW
基础学习率： 0.0004
Warmup 第 1 轮实际学习率： 4e-05
调度器： LambdaLR


## 5. 一个 batch 的反向传播检查

正式训练前只取一个 batch，检查输入、输出、loss 和梯度。这里不调用 `optimizer.step()`，因此不会把这次检查当作正式训练。

In [5]:
cnn.train()
batch_images, batch_labels = next(iter(train_loader))
optimizer.zero_grad(set_to_none=True)
batch_logits = cnn(batch_images)
loss = criterion(batch_logits, batch_labels)
loss.backward()
parameters_with_gradient = sum(
    parameter.grad is not None for parameter in cnn.parameters() if parameter.requires_grad
)
trainable_tensors = sum(parameter.requires_grad for parameter in cnn.parameters())

print("batch 图片：", tuple(batch_images.shape))
print("batch logits：", tuple(batch_logits.shape))
print(f"交叉熵 loss：{loss.item():.4f}")
print(f"获得梯度的参数张量：{parameters_with_gradient}/{trainable_tensors}")
assert batch_logits.shape == (BATCH_SIZE, 10)
assert parameters_with_gradient == trainable_tensors

batch 图片： (128, 3, 32, 32)
batch logits： (128, 10)
交叉熵 loss：2.4509
获得梯度的参数张量：26/26


## 6. 正式训练与输出位置

在 `vision_transformer_practice` 目录运行：

```powershell
conda run -n dl-study python train_cnn_baseline.py
```

训练完成后运行：

```powershell
conda run -n dl-study python evaluate_cnn_baseline.py
```

CNN 输出保存到独立位置：

- checkpoint：`checkpoints/cnn_baseline_best.pt`；
- 训练历史：`results/cnn_baseline/training_history.json`；
- 测试指标和图像：`results/cnn_baseline/`。

TinyViT 原有结果不会被覆盖。

## 7. 公平对照的边界

相同超参数代表控制变量实验，但不代表对两个模型都达到了最佳配置。AdamW、Mixup、Warmup 和较强数据增强最初是围绕 TinyViT 选择的；CNN 可能更适合另一组学习率或正则化。

因此第一轮结论应写成：

> 在相同数据、参数规模和训练方案下，CNN 与 TinyViT 的表现分别如何。

不能直接扩展成“CNN 永远优于 ViT”或“ViT 永远优于 CNN”。公平对照完成以后，才适合分别调参比较各自的最佳能力。

## 8. 本课总结

- CNN 与 TinyViT 接收相同 `B x 3 x 32 x 32` 输入并输出 `B x 10`；
- CNN 参数量为 4,692,426，TinyViT 为 4,771,082，相差约 1.649%；
- 两者复用完全相同的数据、损失函数、优化器、训练循环和评估方法；
- checkpoint 和 results 分开保存，仅用于避免文件覆盖；
- 下一步正式训练 CNN，再生成同规格训练曲线和测试指标。

## 9. 自测问题

1. 为什么不能直接使用只有几万参数的小 CNN 对比当前 TinyViT？
2. 哪些项目属于控制变量，哪些属于网络结构差异？
3. 为什么 CNN 和 TinyViT 必须使用相同训练/验证索引？
4. 为什么两个模型的 checkpoint 必须分开保存？
5. 相同超参数为什么不等于两个模型都处于最佳状态？